In [1]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import statsmodels.api as sm


In [2]:
gdf = gpd.read_file('../../dataset/shp/final_version.shp')
gdf

,CD_FCU,NM_FCU,CD_UF,NM_UF,SIGLA_UF,CD_MUN,NM_MUN,distance,dist_km,dist_norm,...,inp_hosp,wtr_idx,sanit_idx,liv_a_idx,hh_idx,cluster,population,area_km2,density_km,geometry
0,35503080069,Jardim Miliunas,35,São Paulo,SP,3550308,São Paulo,23646.846826,23.646847,0.533159,...,2071.724079,0.997450,0.451028,0.681208,0.321536,1.0,1159,0.026,45062.21,"POLYGON ((-5162009.388 -2692597.734, -5162042...."
1,42024040016,Rua Araranguá,42,Santa Catarina,SC,4202404,Blumenau,2544.012189,2.544012,0.080354,...,1167.193154,1.000000,0.978808,0.872972,0.999800,0.0,340,0.117,2916.00,"POLYGON ((-5460228.537 -3115389.701, -5460231...."
2,33045570554,Tiqui,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,28454.700968,28.454701,0.507651,...,1003.263292,1.000000,1.000000,0.853671,0.999800,0.0,394,0.028,14189.51,"POLYGON ((-4840464.531 -2618564.009, -4840451...."
3,33045570104,Parque Nossa Senhora da Penha,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,1286.545036,1.286545,0.018809,...,659.135306,0.990750,0.892921,0.579645,1.000000,0.0,1058,0.016,65726.53,"POLYGON ((-4810854.603 -2616904.986, -4810851...."
4,35095020135,Núcleo Residencial Jardim das Andorinhas II,35,São Paulo,SP,3509502,Campinas,5245.591377,5.245591,0.146824,...,1488.467146,0.991000,0.699696,0.828075,0.999800,1.0,658,0.037,17788.59,"POLYGON ((-5233560.28 -2622317.23, -5233639.32..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12339,27043020123,Vila Santa Cruz,27,Alagoas,AL,2704302,Maceió,12464.347471,12.464347,0.490099,...,2773.897971,0.854507,0.545140,0.804361,0.999800,1.0,585,0.030,19327.98,"POLYGON ((-3981406.076 -1070408.33, -3981639.1..."
12340,28003080047,Invasão da Jabotiana,28,Sergipe,SE,2800308,Aracaju,5943.694057,5.943694,0.187358,...,2430.944342,0.974801,0.385397,0.844824,0.999800,1.0,334,0.019,17261.87,"POLYGON ((-4128645.906 -1225338.301, -4128644...."
12341,35188000223,Residencial Bambi II,35,São Paulo,SP,3518800,Guarulhos,18639.788181,18.639788,0.949105,...,5287.866697,0.704965,0.532710,0.804146,0.999800,2.0,390,0.101,3868.86,"POLYGON ((-5164614.754 -2677936.544, -5164656...."
12342,26116060881,Sítio Wanderley,26,Pernambuco,PE,2611606,Recife,7352.563603,7.352564,0.570652,...,811.613421,0.930403,0.704766,0.848365,0.997201,1.0,2015,0.104,19323.35,"POLYGON ((-3890716.958 -898474.936, -3890713.9..."


In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

def scaling_analysis(df, pop_col, y_cols, min_pop=None):
    """
    df      : DataFrame com população e variáveis de interesse
    pop_col : nome da coluna de população (ex: 'pop')
    y_cols  : lista de colunas a testar (ex: ['idx_water', 'idx_sanitation'])
    min_pop : opcional, filtra observações com população >= min_pop
    """
    results = []

    for y_col in y_cols:
        data = df[[pop_col, y_col]].copy()
        data = data.replace([np.inf, -np.inf], np.nan).dropna()

        # filtra população mínima (se quiser evitar favelas muito pequenas)
        if min_pop is not None:
            data = data[data[pop_col] >= min_pop]

        # precisa ter valores > 0 para log
        data = data[(data[pop_col] > 0) & (data[y_col] > 0)]
        if len(data) < 10:  # pouco dado, pula
            continue

        logN = np.log(data[pop_col])
        logY = np.log(data[y_col])

        X = sm.add_constant(logN)  # adiciona intercepto
        model = sm.OLS(logY, X).fit()

        alpha = model.params["const"]
        beta = model.params[logN.name]
        r2 = model.rsquared
        p_beta = model.pvalues[logN.name]
        ci_low, ci_high = model.conf_int().loc[logN.name]

        results.append({
            "variable": y_col,
            "alpha": alpha,
            "beta": beta,
            "beta_ci_low": ci_low,
            "beta_ci_high": ci_high,
            "p_beta": p_beta,
            "r2": r2,
            "n_obs": len(data)
        })

    return pd.DataFrame(results)


In [5]:
y_cols = ["wtr_idx", "sanit_idx", "liv_a_idx", "hh_idx"]
scaling_results = scaling_analysis(gdf, pop_col="population", y_cols=y_cols, min_pop=100)
print(scaling_results)


    variable     alpha      beta  beta_ci_low  beta_ci_high        p_beta  \
0    wtr_idx -0.245736  0.016309     0.009763      0.022854  1.055010e-06   
1  sanit_idx -0.545075  0.029100     0.023654      0.034545  1.449119e-25   
2  liv_a_idx -0.156301 -0.010556    -0.012591     -0.008521  3.424192e-24   
3     hh_idx -0.039500  0.003463     0.001479      0.005446  6.256125e-04   

         r2  n_obs  
0  0.001963  12125  
1  0.008969  12126  
2  0.008457  12125  
3  0.000964  12126  
